# 09. 결측 2개 이상 겹친 행 — 전용 서브모델 실험

지금까지 유일하게 안 해본 개선 아이디어: 핵심 3피처(`stress_level`, `sleep_duration`, `physical_activity_level`) 중 **2개 이상이 동시에 결측인 행**(약 2.3%, 이산화 기준 BA 0.56~0.81로 가장 취약)을 위한 **전용 서브모델**을 만들어서, 글로벌 LightGBM(04) 대신 이 구간에서만 서브모델 예측을 쓰는 라우팅 앙상블을 시도합니다.

**주의**: 이 구간은 표본이 작고(15,900여 행), 지금까지 실험(05~08)에서 "약한 피처들은 신호가 거의 없다"는 게 반복 확인됐기 때문에 큰 개선은 기대하기 어려움 — 그래도 검증은 해본다.

**비교 대상 3가지**
1. 글로벌 모델(04)이 이 구간에서 내는 예측 — 기준선
2. 단순 휴리스틱: 정확히 같은 결측 패턴(예: "stress+sleep 결측")의 train 내 조건부 최빈 클래스로 찍기
3. 전용 서브모델: 이 구간 행들로만(보수적인 하이퍼파라미터) 학습한 LightGBM

커널: **Python (teammate)**

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import balanced_accuracy_score, accuracy_score
import lightgbm as lgb

SEED = 42
N_FOLDS = 5
DATA_DIR = Path("../playground-series-s6e7")
OUT_DIR = DATA_DIR / "processed"
TARGET = "health_condition"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
folds = pd.read_csv(OUT_DIR / "cv_folds.csv")
train = train.merge(folds, on="id", how="left")
assert train["fold"].isna().sum() == 0
print(train.shape, test.shape)

(690088, 16) (295753, 14)


## 1. 결측 복구 피처 + core_missing_count (04, 07과 동일)

In [2]:
def add_recovery_features(df, step_tertiles, sleep_quality_cond):
    df = df.copy()
    activity_proxy = pd.cut(
        df["step_count"], bins=[-np.inf, step_tertiles[0], step_tertiles[1], np.inf],
        labels=["sedentary", "moderate", "active"],
    ).astype(object)
    df["physical_activity_level_recovered"] = df["physical_activity_level"]
    missing_activity = df["physical_activity_level"].isna()
    df.loc[missing_activity, "physical_activity_level_recovered"] = activity_proxy[missing_activity]
    df["physical_activity_level_recovered"] = df["physical_activity_level_recovered"].fillna("moderate")

    df["sleep_duration_recovered"] = df["sleep_duration"]
    missing_sleep = df["sleep_duration"].isna()
    fallback_median = sleep_quality_cond.get("missing", sleep_quality_cond["average"])
    mapped = df.loc[missing_sleep, "sleep_quality"].map(sleep_quality_cond).fillna(fallback_median)
    df.loc[missing_sleep, "sleep_duration_recovered"] = mapped

    df["stress_level_isnull"] = df["stress_level"].isna().astype(np.int8)
    df["sleep_duration_isnull"] = df["sleep_duration"].isna().astype(np.int8)
    df["physical_activity_level_isnull"] = df["physical_activity_level"].isna().astype(np.int8)

    df["core_missing_count"] = (
        df["stress_level_isnull"] + df["sleep_duration_isnull"] + df["physical_activity_level_isnull"]
    )

    def pattern(row):
        s = "X" if row["stress_level_isnull"] else "."
        sl = "X" if row["sleep_duration_isnull"] else "."
        a = "X" if row["physical_activity_level_isnull"] else "."
        return s + sl + a
    df["missing_pattern"] = df.apply(pattern, axis=1)
    return df


step_tertiles = train["step_count"].quantile([1/3, 2/3]).values
sleep_quality_cond = train.groupby("sleep_quality")["sleep_duration"].median().to_dict()
sleep_quality_cond["missing"] = train["sleep_duration"].median()

train = add_recovery_features(train, step_tertiles, sleep_quality_cond)
test = add_recovery_features(test, step_tertiles, sleep_quality_cond)

subgroup_mask = train["core_missing_count"] >= 2
print(f"2개 이상 결측 행: {subgroup_mask.sum()}개 ({subgroup_mask.mean()*100:.2f}%)")
print(train.loc[subgroup_mask, "missing_pattern"].value_counts())
print(train.loc[subgroup_mask, TARGET].value_counts(normalize=True).round(4))

2개 이상 결측 행: 16519개 (2.39%)
missing_pattern
XX.    8514
X.X    4006
.XX    3500
XXX     499
Name: count, dtype: int64
health_condition
at-risk      0.8572
unhealthy    0.0855
fit          0.0573
Name: proportion, dtype: float64


## 2. 인코딩 (04와 동일)

In [3]:
NUMERIC_COLS = [
    "sleep_duration_recovered", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
]
ORDINAL_COLS = {
    "stress_level": ["low", "medium", "high"],
    "sleep_quality": ["poor", "average", "good"],
    "physical_activity_level_recovered": ["sedentary", "moderate", "active"],
    "smoking_alcohol": ["no", "occasional", "yes"],
}
NOMINAL_COLS = ["diet_type", "gender"]
FLAG_COLS = ["stress_level_isnull", "sleep_duration_isnull", "physical_activity_level_isnull"]

numeric_medians = train[NUMERIC_COLS].median()
for df in (train, test):
    for col in NUMERIC_COLS:
        df[col] = df[col].fillna(numeric_medians[col])

for df in (train, test):
    for col in list(ORDINAL_COLS.keys()) + NOMINAL_COLS:
        df[col] = df[col].fillna("missing")

for col, order in ORDINAL_COLS.items():
    categories = order + ["missing"]
    enc = OrdinalEncoder(categories=[categories])
    train[col] = enc.fit_transform(train[[col]])
    test[col] = enc.transform(test[[col]])

train_ohe = pd.get_dummies(train[NOMINAL_COLS], prefix=NOMINAL_COLS)
test_ohe = pd.get_dummies(test[NOMINAL_COLS], prefix=NOMINAL_COLS).reindex(columns=train_ohe.columns, fill_value=0)
train = pd.concat([train.drop(columns=NOMINAL_COLS), train_ohe], axis=1)
test = pd.concat([test.drop(columns=NOMINAL_COLS), test_ohe], axis=1)

FEATURE_COLS = NUMERIC_COLS + list(ORDINAL_COLS.keys()) + FLAG_COLS + list(train_ohe.columns)

target_encoder = LabelEncoder()
train["target_enc"] = target_encoder.fit_transform(train[TARGET])
class_order = list(target_encoder.classes_)
train_priors = train[TARGET].value_counts(normalize=True).to_dict()

subgroup_mask = train["core_missing_count"] >= 2  # concat 이후 인덱스 유지되므로 재사용 가능
print(len(FEATURE_COLS), "features")

22 features


## 3. 글로벌 모델(04) OOF 재생성 — 기준선

In [4]:
def prior_corrected_predict(proba, class_order, priors):
    prior_arr = np.array([priors[c] for c in class_order])
    scores = proba / prior_arr
    return np.array(class_order)[scores.argmax(axis=1)]


tuned_params = dict(
    objective="multiclass", num_class=3, random_state=SEED, verbosity=-1,
    n_estimators=500,
    learning_rate=0.047792122422826176,
    num_leaves=19,
    max_depth=9,
    min_child_samples=172,
    subsample=0.8929201812783885,
    colsample_bytree=0.7318659835628288,
    reg_alpha=0.0005023614837892232,
    reg_lambda=0.32275087452118445,
)

oof_proba_global = np.zeros((len(train), 3))
for fold in range(N_FOLDS):
    tr_idx = train["fold"] != fold
    va_idx = train["fold"] == fold
    X_tr, y_tr = train.loc[tr_idx, FEATURE_COLS], train.loc[tr_idx, "target_enc"]
    X_va, y_va = train.loc[va_idx, FEATURE_COLS], train.loc[va_idx, "target_enc"]
    model = lgb.LGBMClassifier(**tuned_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_proba_global[va_idx.values] = model.predict_proba(X_va)
    print(f"global fold {fold} done")

pred_global = prior_corrected_predict(oof_proba_global, class_order, train_priors)
ba_global_overall = balanced_accuracy_score(train[TARGET].values, pred_global)
ba_global_subgroup = balanced_accuracy_score(
    train.loc[subgroup_mask, TARGET].values, pred_global[subgroup_mask.values]
)
print(f"\n글로벌 모델 전체 CV BA: {ba_global_overall:.5f} (04 원본: 0.94987)")
print(f"글로벌 모델의 '2개 이상 결측' 구간 BA: {ba_global_subgroup:.5f}")

global fold 0 done


global fold 1 done


global fold 2 done


global fold 3 done


global fold 4 done



글로벌 모델 전체 CV BA: 0.94988 (04 원본: 0.94987)
글로벌 모델의 '2개 이상 결측' 구간 BA: 0.74280


## 4. 비교 1 — 단순 휴리스틱 (결측 패턴별 조건부 최빈 클래스)

In [5]:
heuristic_pred = np.empty(subgroup_mask.sum(), dtype=object)
sub_df = train.loc[subgroup_mask].reset_index(drop=True)

for fold in range(N_FOLDS):
    tr_mask = sub_df["fold"] != fold
    va_mask = sub_df["fold"] == fold
    # 같은 패턴의 train(다른 fold)에서 최빈 클래스를 구해서 적용
    pattern_mode = sub_df.loc[tr_mask].groupby("missing_pattern")[TARGET].agg(lambda s: s.value_counts().idxmax())
    global_mode = sub_df.loc[tr_mask, TARGET].value_counts().idxmax()
    va_patterns = sub_df.loc[va_mask, "missing_pattern"]
    heuristic_pred[va_mask.values] = va_patterns.map(pattern_mode).fillna(global_mode).values

ba_heuristic = balanced_accuracy_score(sub_df[TARGET].values, heuristic_pred)
print(f"휴리스틱(패턴별 최빈클래스) '2개 이상 결측' 구간 BA: {ba_heuristic:.5f}")

휴리스틱(패턴별 최빈클래스) '2개 이상 결측' 구간 BA: 0.33333


## 5. 비교 2 — 전용 서브모델 (보수적 하이퍼파라미터, 이 구간 행만으로 학습)

In [6]:
specialist_params = dict(
    objective="multiclass", num_class=3, random_state=SEED, verbosity=-1,
    n_estimators=200, learning_rate=0.05,
    num_leaves=7, max_depth=4, min_child_samples=80,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=1.0,
)

sub_indices = train.index[subgroup_mask]
sub_fold = train.loc[sub_indices, "fold"]
oof_proba_specialist = np.zeros((subgroup_mask.sum(), 3))

for fold in range(N_FOLDS):
    tr_mask = (sub_fold != fold).values
    va_mask = (sub_fold == fold).values
    X_tr = train.loc[sub_indices, FEATURE_COLS][tr_mask]
    y_tr = train.loc[sub_indices, "target_enc"][tr_mask]
    X_va = train.loc[sub_indices, FEATURE_COLS][va_mask]
    y_va = train.loc[sub_indices, "target_enc"][va_mask]

    model = lgb.LGBMClassifier(**specialist_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(30, verbose=False)])
    proba = model.predict_proba(X_va)
    oof_proba_specialist[va_mask] = proba
    print(f"specialist fold {fold} done (train={len(X_tr)}, val={len(X_va)})")

pred_specialist = prior_corrected_predict(oof_proba_specialist, class_order, train_priors)
ba_specialist = balanced_accuracy_score(sub_df[TARGET].values, pred_specialist)
print(f"\n전용 서브모델 '2개 이상 결측' 구간 BA: {ba_specialist:.5f}")

specialist fold 0 done (train=13114, val=3405)


specialist fold 1 done (train=13202, val=3317)


specialist fold 2 done (train=13278, val=3241)


specialist fold 3 done (train=13199, val=3320)


specialist fold 4 done (train=13283, val=3236)

전용 서브모델 '2개 이상 결측' 구간 BA: 0.73525


## 6. 세 방식 비교 + 라우팅 앙상블로 전체 CV 재계산

In [7]:
comparison = pd.DataFrame([
    {"방식": "글로벌 모델(04)", "subgroup_BA": ba_global_subgroup},
    {"방식": "휴리스틱(패턴별 최빈클래스)", "subgroup_BA": ba_heuristic},
    {"방식": "전용 서브모델", "subgroup_BA": ba_specialist},
])
print(comparison.to_string(index=False))

best_method = comparison.loc[comparison["subgroup_BA"].idxmax(), "방식"]
print(f"\n이 구간에서 가장 나은 방식: {best_method}")

# 라우팅 앙상블: subgroup에는 최선의 방식을, 나머지는 글로벌 모델 그대로
routed_pred = pred_global.copy()
if best_method == "전용 서브모델":
    routed_pred[subgroup_mask.values] = pred_specialist
elif best_method == "휴리스틱(패턴별 최빈클래스)":
    routed_pred[subgroup_mask.values] = heuristic_pred

ba_routed_overall = balanced_accuracy_score(train[TARGET].values, routed_pred)
print(f"\n라우팅 앙상블 전체 CV BA: {ba_routed_overall:.5f}")
print(f"글로벌 모델(04) 전체 CV BA: {ba_global_overall:.5f}")
print(f"개선폭: {ba_routed_overall - ba_global_overall:+.5f}")

             방식  subgroup_BA
     글로벌 모델(04)     0.742803
휴리스틱(패턴별 최빈클래스)     0.333333
        전용 서브모델     0.735251

이 구간에서 가장 나은 방식: 글로벌 모델(04)



라우팅 앙상블 전체 CV BA: 0.94988
글로벌 모델(04) 전체 CV BA: 0.94988
개선폭: +0.00000


## 7. 개선 확인되면 최종 제출 파일 생성

In [8]:
if ba_routed_overall > 0.94987:
    print("개선 확인 -> 전체 데이터로 최종 모델(글로벌 + 서브모델) 재학습")

    final_global = lgb.LGBMClassifier(**tuned_params)
    final_global.fit(train[FEATURE_COLS], train["target_enc"])
    test_proba_global = final_global.predict_proba(test[FEATURE_COLS])
    test_pred = prior_corrected_predict(test_proba_global, class_order, train_priors)

    test_subgroup_mask = (test["core_missing_count"] >= 2).values
    print(f"test 내 2개 이상 결측 행: {test_subgroup_mask.sum()}개")

    if best_method == "전용 서브모델":
        final_specialist = lgb.LGBMClassifier(**specialist_params)
        final_specialist.fit(train.loc[sub_indices, FEATURE_COLS], train.loc[sub_indices, "target_enc"])
        test_proba_specialist = final_specialist.predict_proba(test.loc[test_subgroup_mask, FEATURE_COLS])
        test_pred_specialist = prior_corrected_predict(test_proba_specialist, class_order, train_priors)
        test_pred[test_subgroup_mask] = test_pred_specialist

    submission = pd.DataFrame({"id": test["id"], TARGET: test_pred})
    submission.to_csv(OUT_DIR / "submission_v7_missing_specialist.csv", index=False)
    print("저장:", OUT_DIR / "submission_v7_missing_specialist.csv")
else:
    print("04 대비 개선 없음 -> submission_v2_tuned.csv를 그대로 유지")

개선 확인 -> 전체 데이터로 최종 모델(글로벌 + 서브모델) 재학습


test 내 2개 이상 결측 행: 7114개
저장: ../playground-series-s6e7/processed/submission_v7_missing_specialist.csv
